In [ ]:
import azure.cognitiveservices.speech as speechsdk

AZURE_SPEECH_KEY = 'AZURE_SPEECH_KEY'
AZURE_SERVICE_REGION = 'REGION'

# Initialize configurations
speech_config = speechsdk.SpeechConfig(subscription=AZURE_SPEECH_KEY, region=AZURE_SERVICE_REGION)

# Set the LanguageIdMode to Continuous
speech_config.set_property(property_id=speechsdk.PropertyId.SpeechServiceConnection_LanguageIdMode, value='Continuous')

conv_speech_config = speechsdk.SpeechConfig(subscription=AZURE_SPEECH_KEY, region=AZURE_SERVICE_REGION)

# Auto language detection configuration
auto_detect_source_language_config = speechsdk.languageconfig.AutoDetectSourceLanguageConfig(languages=["en-SG", "zh-CN", "ms-MY", "hi-IN"])

def speech_recognizer_recognition_canceled_cb(evt: speechsdk.SessionEventArgs):
    print('Canceled event')

def speech_recognizer_session_stopped_cb(evt: speechsdk.SessionEventArgs):
    print('SessionStopped event')

def speech_recognizer_recognized_cb(evt: speechsdk.SpeechRecognitionEventArgs):
    print('TRANSCRIBED_SR:')
    if evt.result.reason == speechsdk.ResultReason.RecognizedSpeech:
        print(f'\tText={evt.result.text}')
    elif evt.result.reason == speechsdk.ResultReason.NoMatch:
        print('\tNOMATCH: Speech could not be TRANSCRIBED: {}'.format(evt.result.no_match_details))

def conversation_transcriber_recognition_canceled_cb(evt: speechsdk.SessionEventArgs):
    print('Canceled event')

def conversation_transcriber_session_stopped_cb(evt: speechsdk.SessionEventArgs):
    print('SessionStopped event')

def conversation_transcriber_transcribed_cb(evt: speechsdk.SpeechRecognitionEventArgs):
    print('TRANSCRIBED:')
    if evt.result.reason == speechsdk.ResultReason.RecognizedSpeech:
        print('\tText={}'.format(evt.result.text))
        print('\tSpeaker ID={}'.format(evt.result.speaker_id))
    elif evt.result.reason == speechsdk.ResultReason.NoMatch:
        print('\tNOMATCH: Speech could not be TRANSCRIBED: {}'.format(evt.result.no_match_details))

def conversation_transcriber_session_started_cb(evt: speechsdk.SessionEventArgs):
    print('SessionStarted event')        

def speech_recognizer_recognizing_cb(evt: speechsdk.SpeechRecognitionEventArgs):
    if evt.result.reason == speechsdk.ResultReason.RecognizingSpeech:
        print(f'RECOGNIZING: {evt.result.text}')

def speech_recognizer_session_started_cb(evt: speechsdk.SessionEventArgs):
    print('SessionStarted event')

# Initialize recognizer
audio_config = speechsdk.audio.AudioConfig(use_default_microphone=True)
speech_recognizer = speechsdk.SpeechRecognizer(
    speech_config=speech_config,
    auto_detect_source_language_config=auto_detect_source_language_config,
    audio_config=audio_config
)

conversation_transcriber = speechsdk.transcription.ConversationTranscriber(
    speech_config=speech_config,
    audio_config=audio_config,
    auto_detect_source_language_config=auto_detect_source_language_config
)
transcribing_stop = False

def stop_cb(evt: speechsdk.SessionEventArgs):
    # Callback to stop continuous recognition upon receiving an event `evt`
    print('CLOSING on {}'.format(evt))
    global transcribing_stop
    transcribing_stop = True

# Connect callbacks
speech_recognizer.recognized.connect(speech_recognizer_recognized_cb)
speech_recognizer.recognizing.connect(speech_recognizer_recognizing_cb)
speech_recognizer.session_started.connect(speech_recognizer_session_started_cb)
speech_recognizer.session_stopped.connect(speech_recognizer_session_stopped_cb)
speech_recognizer.canceled.connect(speech_recognizer_recognition_canceled_cb)

conversation_transcriber.transcribed.connect(conversation_transcriber_transcribed_cb)
conversation_transcriber.session_started.connect(conversation_transcriber_session_started_cb)
conversation_transcriber.session_stopped.connect(conversation_transcriber_session_stopped_cb)
conversation_transcriber.canceled.connect(conversation_transcriber_recognition_canceled_cb)


# Start continuous recognition
print("Starting continuous recognition...")
speech_recognizer.start_continuous_recognition_async()
#conversation_transcriber.start_transcribing_async()

transcribing_stop = False



# Keep the program running to listen to the microphone
import time
try:
    while True:
        time.sleep(0.1)
           
except KeyboardInterrupt:
    print("KeyBorad Stopping continuous recognition...")
    speech_recognizer.stop_continuous_recognition_async()
    #conversation_transcriber.session_stopped.connect(stop_cb)
    #conversation_transcriber.canceled.connect(stop_cb)
    #conversation_transcriber.stop_transcribing_async() 
    print("Recognition stopped.")